In [ ]:
def optimize_sigma_lambda(data=None,target=None,split=2,kernel='Laplacian',min_sigma=1,step=1000,max_sigma=10000,shuffle=True):
    '''
    Optimize sigma is a function that optimises the sigma and lambda hyperparameter values. 
    '''
    
    import pandas as pd
    import numpy as np
    import glob
    import sys
    from qml.math import cho_solve
    from qml.kernels import *
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import r2_score
    from natsort import natsorted
    from sklearn.metrics import mean_squared_error
    from sklearn.model_selection import KFold
    
    eta = np.logspace(-10, 0, 5)
    ert_mae=[]
    ert_nr=[]
    for t in eta:
        Z=pd.concat([pd.DataFrame(data),pd.DataFrame(target)],axis=1)
        kf = KFold(n_splits=split, shuffle=shuffle,random_state=137)
        kf.get_n_splits(Z)
        tab=[]
        for train_index, test_index in kf.split(Z):
            mae=[]
            nr=[]
            X_train = Z.iloc[train_index].drop(list(Z.iloc[train_index].iloc[:,-1:]),axis=1)
            X_test = Z.iloc[test_index].drop(list(Z.iloc[test_index].iloc[:,-1:]),axis=1)
            y_train = Z.iloc[train_index].iloc[:,-1:][list(Z.iloc[train_index].iloc[:,-1:])[0]]
            y_test = Z.iloc[test_index].iloc[:,-1:][list(Z.iloc[test_index].iloc[:,-1:])[0]]
            for i in range(min_sigma,max_sigma,step):
                if kernel == 'Laplacian':
                    K=laplacian_kernel(X_train,X_train,i)
                    K[np.diag_indices_from(K)] +=t
                    v=np.mean(np.abs(np.dot(laplacian_kernel(X_test,X_train,i),cho_solve(K,y_train))-y_test))
                else:
                    K=gaussian_kernel(X_train,X_train,i)
                    K[np.diag_indices_from(K)] +=t
                    v=np.mean(np.abs(np.dot(gaussian_kernel(X_test,X_train,i),cho_solve(K,y_train))-y_test))
                mae.append(v)
                nr.append(i)
            A=pd.DataFrame(mae,columns=['mae'])
            B=pd.DataFrame(nr,columns=['Nr'])
            C=pd.concat([A,B],axis=1)
            tab.append(C)
        ert_mae.append((sum(tab)/len(tab)).loc[(sum(tab)/len(tab))['mae'] == (sum(tab)/len(tab))['mae'].min()]['mae'])
        ert_nr.append((sum(tab)/len(tab)).loc[(sum(tab)/len(tab))['mae'] == (sum(tab)/len(tab))['mae'].min()]['Nr'])
    return (ert_mae, ert_nr,eta)

In [ ]:
def kernel_ridge_regression(target, input_data, kernel='Laplacian', step=900, test_size=0.2, sigma=1, lambd=1e-5):
    '''
    Function that generates the LC together with the MAE, RMSE and R2 for each training points
    '''
    import pandas as pd
    import numpy as np
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import r2_score, mean_squared_error
    from qml.math import cho_solve
    from qml.kernels import *
    
    results = {"Nr": [], "mae": [], "std_MAE": [], "r2": [], "std_R2": [], "rmse": [], "std_RMSE": []}

    for i in range(100, int(input_data.shape[0] * (1 - test_size)), step):
        mae_vals, r2_vals, rmse_vals = [], [], []

        for _ in range(5): # 5-fold shuffling
            X_train, X_test, y_train, y_test = train_test_split(input_data, target, test_size=test_size, shuffle=True)
            if kernel == 'Laplacian':
                K = laplacian_kernel(X_train[:i], X_train[:i], sigma)
                Ks = laplacian_kernel(X_test, X_train[:i], sigma)
            else:
                K = gaussian_kernel(X_train[:i], X_train[:i], sigma)
                Ks = gaussian_kernel(X_test, X_train[:i], sigma)

            K[np.diag_indices_from(K)] += lambd
            alpha = cho_solve(K, y_train[:i])
            y_pred = Ks @ alpha

            mae_vals.append(np.mean(np.abs(y_pred - y_test)))
            r2_vals.append(r2_score(y_test, y_pred))
            rmse_vals.append(mean_squared_error(y_test, y_pred, squared=False))

        results["Nr"].append(i)
        results["mae"].append(np.mean(mae_vals))
        results["std_MAE"].append(np.std(mae_vals))
        results["r2"].append(np.mean(r2_vals))
        results["std_R2"].append(np.std(r2_vals))
        results["rmse"].append(np.mean(rmse_vals))
        results["std_RMSE"].append(np.std(rmse_vals))

    return pd.DataFrame(results)

In [ ]:
from joblib import Parallel, delayed
import pandas as pd
import numpy as np
import glob
from qml.math import cho_solve
from qml.kernels import *
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from natsort import natsorted

def KRR_EHO(number,path=None,sigma=None,lambd=None,path_results=None):
    '''
    Kernel Ridge Regression with Elemental Hold-Out (writen only for the laplacian kernel)
    '''
    ele_ho_data={}
    ele_ho_prop={}
    q=0
    for i in natsorted(glob.glob(path+'/*.npy')):
        ele_ho_data[q]=np.load(i)
        ele_ho_prop[q]=pd.read_csv(i.split('.')[0]+'.csv')
        q=q+1
    x=ele_ho_data.pop(number)
    y=ele_ho_prop.pop(number)
    X=np.concatenate(([ele_ho_data[x] for x in list(ele_ho_data)]),axis=0)
    Y=pd.concat([ele_ho_prop[x] for x in list(ele_ho_prop)],axis=0)
    Y=Y['GAP']
    y=y['GAP']
    K=laplacian_kernel(X,X,sigma)
    K[np.diag_indices_from(K)] +=lambd
    alpha=cho_solve(K,Y)
    Ks=laplacian_kernel(x,X,sigma)
    y_pred=np.dot(Ks,alpha)
    results=[]
    results.append(np.mean(np.abs(y_pred-y)))
    results.append(mean_absolute_error(y,y_pred))
    results.append(r2_score(y,y_pred))
    results.append(mean_squared_error(y, y_pred, squared=False))
    A=pd.DataFrame(results)
    A.to_csv(path_results+str(number)+'.csv')
    return

results = Parallel(n_jobs=19)(delayed(KRR_EHO)(q) for q in [x for x in range(0,19,1)])